In [ ]:
!pip install sqlalchemy_mate==2.0.0.0

In [ ]:
!pip install uszipcode

In [ ]:
from uszipcode import SearchEngine
search = SearchEngine()

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/CHAPA_Chapter-40B_Application-Data_2021-2025_merged_v0.1 - CHAPA_Chapter-40B_Application-Data_2021-2025_merged_v0.1 (1).csv')

In [ ]:
df['property_full_address'] = df['property_street_address'].astype(str) + ', ' + df['property_town_city'].astype(str) + ', ' + df['property_state'].astype(str)

In [ ]:
df['property_full_address'].head(15)

,property_full_address
0,"8 Ciderpress Way, North Andover, MA"
1,"8 Ciderpress Way, North Andover, MA"
2,"8 Ciderpress Way, North Andover, MA"
3,"8 Ciderpress Way, North Andover, MA"
4,"8 Ciderpress Way, North Andover, MA"
5,"8 Ciderpress Way, North Andover, MA"
6,"1301 Albion Road, Bedford, MA"
7,"1301 Albion Road, Bedford, MA"
8,"1301 Albion Road, Bedford, MA"
9,"1301 Albion Road, Bedford, MA"


In [ ]:
!pip install geopy

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1739 entries, 0 to 1738
Data columns (total 31 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   id_number                    1739 non-null   int64  
 1   submission_date              1739 non-null   object 
 2   age                          1739 non-null   int64  
 3   disability                   1739 non-null   int64  
 4   chapa_races                  1739 non-null   object 
 5   hispanic_latino              380 non-null    float64
 6   census_races                 1739 non-null   object 
 7   current_residence_town_city  1739 non-null   object 
 8   current_residence_state      1739 non-null   object 
 9   current_residence_zip        0 non-null      float64
 10  application_source           1739 non-null   object 
 11  hh_size                      1739 non-null   int64  
 12  dependents                   1621 non-null   float64
 13  hh_type           

# property zip, lat, lon, population

In [ ]:
property_lat = [0 for i in range(len(df))]
property_lon = [0 for i in range(len(df))]
property_town_population = [0 for i in range(len(df))]

In [ ]:
town_missing = []

for i in range(len(df)):

  town_name = df['property_town_city'][i]
  state_abbr = df['property_state'][i]

  results = search.by_city_and_state(city=town_name, state=state_abbr, returns=0)

  if results:
    sorted_zips = sorted(
        results,
        key = lambda x : x.population if x.population is not None else -1,
        reverse = True
    )

    selected_zip = sorted_zips[0]

    zip_code = selected_zip.zipcode
    latitude = selected_zip.lat
    longitude = selected_zip.lng
    population = selected_zip.population

    df['property_zip'][i] = zip_code
    property_lat[i] = latitude
    property_lon[i] = longitude
    property_town_population[i] = population

    print(f"--- {i}. {town_name}, {state_abbr} ({len(results)})---")
    print(f"ZIP: {zip_code} | Lat/Lon: {latitude}, {longitude} | Population: {population}")

  else:
    print(f"'{i}, {town_name}, {state_abbr}' could not be find")
    town_missing.append(i)

/tmp/ipython-input-85318812.py:24: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df['property_zip'][i] = zip_code
/tmp/ipython-input-85318812.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame



--- 0. North Andover, MA (1)---
ZIP: 01845 | Lat/Lon: 42.7, -71.11 | Population: 28352
--- 1. North Andover, MA (1)---
ZIP: 01845 | Lat/Lon: 42.7, -71.11 | Population: 28352
--- 2. North Andover, MA (1)---
ZIP: 01845 | Lat/Lon: 42.7, -71.11 | Population: 28352
--- 3. North Andover, MA (1)---
ZIP: 01845 | Lat/Lon: 42.7, -71.11 | Population: 28352
--- 4. North Andover, MA (1)---
ZIP: 01845 | Lat/Lon: 42.7, -71.11 | Population: 28352
--- 5. North Andover, MA (1)---
ZIP: 01845 | Lat/Lon: 42.7, -71.11 | Population: 28352
--- 6. Bedford, MA (1)---
ZIP: 01730 | Lat/Lon: 42.48, -71.26 | Population: 13221
--- 7. Bedford, MA (1)---
ZIP: 01730 | Lat/Lon: 42.48, -71.26 | Population: 13221
--- 8. Bedford, MA (1)---
ZIP: 01730 | Lat/Lon: 42.48, -71.26 | Population: 13221
--- 9. Bedford, MA (1)---
ZIP: 01730 | Lat/Lon: 42.48, -71.26 | Population: 13221
--- 10. Bedford, MA (1)---
ZIP: 01730 | Lat/Lon: 42.48, -71.26 | Population: 13221
--- 11. Taunton, MA (1)---
ZIP: 02780 | Lat/Lon: 41.9, -71.09 | Pop

In [ ]:
df['property_lat'] = property_lat
df['property_lon'] = property_lon
df['property_town_population'] = property_town_population

# currnet zip, lat, lon, population

In [ ]:
current_lat = [0 for i in range(len(df))]
current_lon = [0 for i in range(len(df))]
current_town_population = [0 for i in range(len(df))]

In [ ]:
print(df['current_residence_town_city'][496])
print(df['current_residence_town_city'][497])
print(df['current_residence_town_city'][1297])
print(df['current_residence_town_city'][1723])

Pelham
Pelham
Pelham
Bristol County


In [ ]:
town_drop = [496, 497, 1297, 1723]
town_missing = []

for i in range(len(df)):
  if i in town_drop:
    continue

  town_name = df['current_residence_town_city'][i]
  state_abbr = df['current_residence_state'][i]

  if town_name == 'Unknown':
    continue

  results = search.by_city_and_state(city=town_name, state=state_abbr, returns=0)

  if results:
    sorted_zips = sorted(
        results,
        key = lambda x : x.population if x.population is not None else -1,
        reverse = True
    )

    selected_zip = sorted_zips[0]

    zip_code = selected_zip.zipcode
    latitude = selected_zip.lat
    longitude = selected_zip.lng
    population = selected_zip.population

    df['current_residence_zip'][i] = zip_code
    current_lat[i] = latitude
    current_lon[i] = longitude
    current_town_population[i] = population

    print(f"--- {i}. {town_name}, {state_abbr} ({len(results)})---")
    print(f"ZIP: {zip_code} | Lat/Lon: {latitude}, {longitude} | Population: {population}")

  else:
    print(f"'{i}, {town_name}, {state_abbr}' could not be find")
    town_missing.append(i)

--- 0. Somerville, MA (3)---
ZIP: 02145 | Lat/Lon: 42.39, -71.1 | Population: 25439


/tmp/ipython-input-3176788376.py:30: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df['current_residence_zip'][i] = zip_code
/tmp/ipython-input-3176788376.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from 

--- 1. Westford, MA (1)---
ZIP: 01886 | Lat/Lon: 42.58, -71.43 | Population: 21951
--- 2. Malden, MA (1)---
ZIP: 02148 | Lat/Lon: 42.43, -71.05 | Population: 59503
--- 3. Haverhill, MA (3)---
ZIP: 01830 | Lat/Lon: 42.78, -71.08 | Population: 25137
--- 4. Nashua, NH (4)---
ZIP: 03060 | Lat/Lon: 42.74, -71.46 | Population: 29357
--- 5. East Boston, MA (1)---
ZIP: 02128 | Lat/Lon: 42.36, -71.01 | Population: 40508
--- 6. Waltham, MA (3)---
ZIP: 02453 | Lat/Lon: 42.37, -71.24 | Population: 28968
--- 7. Woburn, MA (1)---
ZIP: 01801 | Lat/Lon: 42.48, -71.15 | Population: 38903
--- 8. Woburn, MA (1)---
ZIP: 01801 | Lat/Lon: 42.48, -71.15 | Population: 38903
--- 9. Burlington, MA (1)---
ZIP: 01803 | Lat/Lon: 42.5, -71.2 | Population: 24487
--- 10. Chelmsford, MA (1)---
ZIP: 01824 | Lat/Lon: 42.59, -71.36 | Population: 24911
--- 11. Mansfield, MA (1)---
ZIP: 02048 | Lat/Lon: 42.02, -71.21 | Population: 23184
--- 12. Taunton, MA (1)---
ZIP: 02780 | Lat/Lon: 41.9, -71.09 | Population: 49036
--- 1

In [ ]:
df['current_lat'] = current_lat
df['current_lon'] = current_lon
df['current_town_population'] = current_town_population

In [ ]:
df.head(10)

,id_number,submission_date,age,disability,chapa_races,hispanic_latino,census_races,current_residence_town_city,current_residence_state,current_residence_zip,...,matched_address,fthb_class,msa,property_restriction,property_lat,property_lon,property_town_population,current_lat,current_lon,current_town_population
0,150733,2022-02-08,65,0,White/Non-Minority,0.0,White alone,Somerville,MA,02145,...,"8 CIDERPRESS WAY, NORTH ANDOVER, MA, 01845",0,"Boston-Cambridge-Newton, MA-NH",NaN,42.70,-71.11,28352,42.39,-71.10,25439
1,439181,2022-02-08,59,0,Hispanic/Latino,1.0,Unknown,Westford,MA,01886,...,"8 CIDERPRESS WAY, NORTH ANDOVER, MA, 01845",0,"Boston-Cambridge-Newton, MA-NH",NaN,42.70,-71.11,28352,42.58,-71.43,21951
2,697399,2022-02-08,0,0,Black or African American,0.0,Black or African American alone,Malden,MA,02148,...,"8 CIDERPRESS WAY, NORTH ANDOVER, MA, 01845",0,"Boston-Cambridge-Newton, MA-NH",NaN,42.70,-71.11,28352,42.43,-71.05,59503
3,191565,2022-02-08,0,0,White/Non-Minority,0.0,White alone,Haverhill,MA,01830,...,"8 CIDERPRESS WAY, NORTH ANDOVER, MA, 01845",0,"Boston-Cambridge-Newton, MA-NH",NaN,42.70,-71.11,28352,42.78,-71.08,25137
4,226436,2022-02-08,60,0,White/Non-Minority,0.0,White alone,Nashua,NH,03060,...,"8 CIDERPRESS WAY, NORTH ANDOVER, MA, 01845",0,"Boston-Cambridge-Newton, MA-NH",NaN,42.70,-71.11,28352,42.74,-71.46,29357
5,932459,2022-02-08,0,0,White/Non-Minority,0.0,White alone,East Boston,MA,02128,...,"8 CIDERPRESS WAY, NORTH ANDOVER, MA, 01845",0,"Boston-Cambridge-Newton, MA-NH",NaN,42.70,-71.11,28352,42.36,-71.01,40508
6,170029,2021-06-15,35,0,White/Non-Minority,0.0,White alone,Waltham,MA,02453,...,"1301 ALBION RD, BEDFORD, MA, 01730",0,"Boston-Cambridge-Newton, MA-NH",NaN,42.48,-71.26,13221,42.37,-71.24,28968
7,725305,2021-06-15,0,0,White/Non-Minority,0.0,White alone,Woburn,MA,01801,...,"1301 ALBION RD, BEDFORD, MA, 01730",0,"Boston-Cambridge-Newton, MA-NH",NaN,42.48,-71.26,13221,42.48,-71.15,38903
8,569863,2021-06-15,38,0,Black or African American,0.0,Black or African American alone,Woburn,MA,01801,...,"1301 ALBION RD, BEDFORD, MA, 01730",0,"Boston-Cambridge-Newton, MA-NH",NaN,42.48,-71.26,13221,42.48,-71.15,38903
9,746052,2021-06-15,63,0,Asian/Pacific Islander,0.0,Asian alone,Burlington,MA,01803,...,"1301 ALBION RD, BEDFORD, MA, 01730",0,"Boston-Cambridge-Newton, MA-NH",NaN,42.48,-71.26,13221,42.50,-71.20,24487


In [ ]:
df.to_csv("/content/drive/MyDrive/Colab Notebooks/total_zip_crawling.csv", index=False)